# Foundational Statistics & Machine Learning Concepts

The topics below are organized into four categories and placed before the original probability-distribution and ML function-family material. Each topic includes a Python function/formula followed by an **SDK visualization** cell.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


## Descriptive Statistics

Summarize the center, frequency, and spread of observed data.

### Mean


In [ ]:
def mean(values):
    if not values: raise ValueError("values cannot be empty")
    return sum(values) / len(values)

sample = [2, 4, 4, 6, 9]
print("Mean:", mean(sample))

**SDK visualization**

In [ ]:
x = np.array([2, 4, 4, 6, 9])
plt.figure(figsize=(6, 3))
plt.scatter(x, np.zeros_like(x), s=90, label="observations")
plt.axvline(x.mean(), color="crimson", label=f"mean = {x.mean():.2f}")
plt.yticks([]); plt.xlabel("Value"); plt.title("Mean as the balance point"); plt.legend(); plt.show()

### Median


In [ ]:
def median(values):
    if not values: raise ValueError("values cannot be empty")
    ordered = sorted(values)
    n = len(ordered); mid = n // 2
    return ordered[mid] if n % 2 else (ordered[mid-1] + ordered[mid]) / 2

sample = [2, 4, 4, 6, 15]
print("Median:", median(sample))

**SDK visualization**

In [ ]:
x = np.array([2, 4, 4, 6, 15])
plt.figure(figsize=(6, 3)); plt.scatter(x, np.zeros_like(x), s=90)
plt.axvline(np.median(x), color="crimson", label=f"median = {np.median(x):.1f}")
plt.yticks([]); plt.xlabel("Value"); plt.title("Median is the middle observation"); plt.legend(); plt.show()

### Mode


In [ ]:
from collections import Counter
def mode(values):
    if not values: raise ValueError("values cannot be empty")
    counts = Counter(values); highest = max(counts.values())
    return sorted([v for v, count in counts.items() if count == highest])

sample = [1, 2, 2, 3, 3, 4]
print("Mode(s):", mode(sample))

**SDK visualization**

In [ ]:
x = np.array([1, 2, 2, 3, 3, 4])
values, counts = np.unique(x, return_counts=True)
plt.figure(figsize=(6, 3)); plt.bar(values, counts)
plt.xlabel("Value"); plt.ylabel("Frequency"); plt.title("Mode: highest frequency"); plt.show()

### Variance


In [ ]:
def variance(values):
    if not values: raise ValueError("values cannot be empty")
    mu = sum(values) / len(values)
    return sum((x - mu)**2 for x in values) / len(values)

sample = [2, 4, 4, 6, 9]
print("Population variance:", variance(sample))

**SDK visualization**

In [ ]:
x = np.array([2, 4, 4, 6, 9]); mu = x.mean()
plt.figure(figsize=(6, 3)); plt.bar(range(len(x)), (x-mu)**2)
plt.axhline(np.var(x), color="crimson", label=f"variance = {np.var(x):.2f}")
plt.xlabel("Observation"); plt.ylabel("Squared deviation"); plt.title("Variance from squared deviations"); plt.legend(); plt.show()

### Standard Deviation


In [ ]:
def standard_deviation(values):
    return math.sqrt(variance(values))

sample = [2, 4, 4, 6, 9]
print("Population standard deviation:", standard_deviation(sample))

**SDK visualization**

In [ ]:
x = np.array([2, 4, 4, 6, 9]); mu, sigma = x.mean(), x.std()
plt.figure(figsize=(6, 3)); plt.hist(x, bins=np.arange(1, 11), alpha=.7, edgecolor="black")
plt.axvline(mu, color="crimson", label=f"mean = {mu:.2f}")
plt.axvspan(mu-sigma, mu+sigma, alpha=.2, label=f"±1 SD = {sigma:.2f}")
plt.xlabel("Value"); plt.ylabel("Count"); plt.title("Spread measured by standard deviation"); plt.legend(); plt.show()

## Probability & Distributions

Represent uncertainty, update beliefs with evidence, and describe data distributions.

### Probability


In [ ]:
def probability(favorable, total):
    if total <= 0 or favorable < 0 or favorable > total:
        raise ValueError("require 0 <= favorable <= total and total > 0")
    return favorable / total

print("P(rolling a 4 or higher on a fair die):", probability(3, 6))

**SDK visualization**

In [ ]:
outcomes = np.arange(1, 7); p = np.where(outcomes >= 4, 1/6, 0)
plt.figure(figsize=(6, 3)); plt.bar(outcomes, p)
plt.xlabel("Die outcome"); plt.ylabel("Probability contribution"); plt.title("Favorable outcomes: 4, 5, 6"); plt.show()

### Bayes’ Theorem


In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    evidence = likelihood*prior + false_positive_rate*(1-prior)
    return likelihood*prior/evidence

print("Posterior:", bayes(prior=.01, likelihood=.95, false_positive_rate=.05))

**SDK visualization**

In [ ]:
prior, sensitivity, fpr = .01, .95, .05
posterior = bayes(prior, sensitivity, fpr)
labels = ["Prior P(A)", "Posterior P(A|B)"]
plt.figure(figsize=(6, 3)); plt.bar(labels, [prior, posterior])
plt.ylim(0, 1); plt.ylabel("Probability"); plt.title(f"Bayesian update: posterior = {posterior:.1%}"); plt.show()

### Normal Distribution


In [ ]:
def normal_pdf(x, mu=0.0, sigma=1.0):
    if sigma <= 0: raise ValueError("sigma must be positive")
    return math.exp(-0.5*((x-mu)/sigma)**2)/(sigma*math.sqrt(2*math.pi))

print("Standard normal density at 0:", normal_pdf(0))

**SDK visualization**

In [ ]:
x=np.linspace(-4,4,400); y=np.exp(-x**2/2)/math.sqrt(2*math.pi)
plt.figure(figsize=(6,3)); plt.plot(x,y); plt.fill_between(x,y,alpha=.2)
plt.xlabel("x"); plt.ylabel("Density"); plt.title("Standard normal distribution (μ=0, σ=1)"); plt.show()

## Relationships & Statistical Inference

Measure relationships and use sample data to evaluate claims about populations.

### Correlation


In [ ]:
def pearson_correlation(x, y):
    if len(x) != len(y) or len(x) < 2: raise ValueError("equal-length inputs; at least 2 values")
    mx, my = sum(x)/len(x), sum(y)/len(y)
    numerator = sum((a-mx)*(b-my) for a,b in zip(x,y))
    denominator = math.sqrt(sum((a-mx)**2 for a in x)*sum((b-my)**2 for b in y))
    if denominator == 0: raise ValueError("correlation undefined for constant input")
    return numerator/denominator

x=[1,2,3,4,5]; y=[2,3,5,4,7]
print("Pearson r:", pearson_correlation(x,y))

**SDK visualization**

In [ ]:
x=np.array([1,2,3,4,5]); y=np.array([2,3,5,4,7]); r=np.corrcoef(x,y)[0,1]
plt.figure(figsize=(5, 4)); plt.scatter(x,y); plt.title(f"Correlation (r = {r:.2f})")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(alpha=.25); plt.show()

### Regression


In [ ]:
def linear_regression(x, y):
    mx, my = sum(x)/len(x), sum(y)/len(y)
    slope = sum((a-mx)*(b-my) for a,b in zip(x,y))/sum((a-mx)**2 for a in x)
    return my-slope*mx, slope

x=[1,2,3,4,5]; y=[2,3,5,4,7]
print("intercept, slope:", linear_regression(x,y))

**SDK visualization**

In [ ]:
x=np.array([1,2,3,4,5]); y=np.array([2,3,5,4,7]); b0,b1=linear_regression(x.tolist(),y.tolist())
xx=np.linspace(x.min(),x.max(),100)
plt.figure(figsize=(5,4)); plt.scatter(x,y,label="observed"); plt.plot(xx,b0+b1*xx,color="crimson",label="least-squares fit")
plt.xlabel("X"); plt.ylabel("Y"); plt.title("Simple linear regression"); plt.legend(); plt.show()

### Hypothesis Testing


In [ ]:
def hypothesis_decision(p_value, alpha=0.05):
    if not 0 <= p_value <= 1: raise ValueError("p_value must be in [0, 1]")
    return "Reject H0" if p_value < alpha else "Fail to reject H0"

print(hypothesis_decision(p_value=.03, alpha=.05))

**SDK visualization**

In [ ]:
alpha=.05; pvals=np.linspace(0,1,201)
plt.figure(figsize=(6,3)); plt.plot(pvals, pvals*0+1); plt.axvspan(0,alpha,ymin=0,ymax=1,color="tomato",alpha=.25,label="reject region (p < α)")
plt.axvline(.03,color="crimson",label="example p = 0.03"); plt.yticks([]); plt.xlabel("p-value"); plt.title("Decision threshold for a hypothesis test"); plt.legend(); plt.show()

### p-Value


In [ ]:
def two_sided_z_pvalue(z):
    return math.erfc(abs(z)/math.sqrt(2))

print("Two-sided p-value for z=2:", two_sided_z_pvalue(2))

**SDK visualization**

In [ ]:
z=np.linspace(-4,4,500); density=np.exp(-z**2/2)/math.sqrt(2*math.pi); cutoff=2
plt.figure(figsize=(6,3)); plt.plot(z,density); plt.fill_between(z,density,where=(z<=-cutoff)|(z>=cutoff),alpha=.35,label="two-sided p-value area")
plt.axvline(-cutoff,color="gray",ls="--"); plt.axvline(cutoff,color="gray",ls="--")
plt.xlabel("z statistic"); plt.ylabel("Density"); plt.title(f"Two-sided p-value = {two_sided_z_pvalue(2):.4f}"); plt.legend(); plt.show()

### Confidence Interval


In [ ]:
def mean_confidence_interval(values, confidence=0.95):
    n=len(values); m=sum(values)/n
    s=math.sqrt(sum((v-m)**2 for v in values)/(n-1))
    critical=stats.t.ppf((1+confidence)/2, df=n-1)
    margin=critical*s/math.sqrt(n)
    return m-margin, m+margin

sample=[12,14,15,13,16,14,15,17]
print("95% CI:", mean_confidence_interval(sample))

**SDK visualization**

In [ ]:
rng=np.random.default_rng(7); population=rng.normal(15,3, size=10000); intervals=[]
for _ in range(40): intervals.append(mean_confidence_interval(rng.choice(population, size=30).tolist()))
plt.figure(figsize=(6,4));
for i,(lo,hi) in enumerate(intervals): plt.plot([lo,hi],[i,i],color="steelblue")
plt.axvline(population.mean(),color="crimson",label="population mean"); plt.xlabel("Interval for sample mean"); plt.ylabel("Repeated sample"); plt.title("Illustrative 95% confidence intervals"); plt.legend(); plt.show()

### T-Test


In [ ]:
def one_sample_t_test(values, mu0=0):
    n=len(values); m=sum(values)/n
    s=math.sqrt(sum((v-m)**2 for v in values)/(n-1))
    t=(m-mu0)/(s/math.sqrt(n)); p=2*(1-stats.t.cdf(abs(t),df=n-1))
    return t,p

print("t statistic, two-sided p:", one_sample_t_test([5,6,7,8,9], mu0=6))

**SDK visualization**

In [ ]:
sample=np.array([5,6,7,8,9]); t,p=one_sample_t_test(sample.tolist(),6);
x=np.linspace(-4,4,400); y=stats.t.pdf(x,df=len(sample)-1)
plt.figure(figsize=(6,3)); plt.plot(x,y); plt.fill_between(x,y,where=np.abs(x)>=abs(t),alpha=.3,label="two-sided tail area")
plt.axvline(t,color="crimson",ls="--",label=f"t = {t:.2f}"); plt.xlabel("t statistic"); plt.title(f"One-sample t-test (p = {p:.4f})"); plt.legend(); plt.show()

### ANOVA


In [ ]:
def one_way_anova(*groups):
    k=len(groups); all_values=[v for g in groups for v in g]; n=len(all_values)
    grand=sum(all_values)/n
    ss_between=sum(len(g)*(sum(g)/len(g)-grand)**2 for g in groups)
    ss_within=sum(sum((v-sum(g)/len(g))**2 for v in g) for g in groups)
    dfb=k-1; dfw=n-k; f=(ss_between/dfb)/(ss_within/dfw)
    return f, stats.f.sf(f,dfb,dfw)

print("F statistic, p-value:", one_way_anova([4,5,6,5],[7,8,9,8],[5,6,5,7]))

**SDK visualization**

In [ ]:
groups=[np.array([4,5,6,5]),np.array([7,8,9,8]),np.array([5,6,5,7])]; f,p=one_way_anova(*[g.tolist() for g in groups])
plt.figure(figsize=(6,3)); plt.boxplot(groups,tick_labels=["A","B","C"]); plt.ylabel("Measurement"); plt.title(f"One-way ANOVA (F={f:.2f}, p={p:.4f})"); plt.show()

### Chi-Square Test


In [ ]:
def chi_square_gof(observed, expected):
    if len(observed)!=len(expected) or any(e<=0 for e in expected): raise ValueError("invalid expected counts")
    statistic=sum((o-e)**2/e for o,e in zip(observed,expected))
    return statistic, stats.chi2.sf(statistic,df=len(observed)-1)

print("chi-square, p-value:", chi_square_gof([18,22,27,33],[25,25,25,25]))

**SDK visualization**

In [ ]:
observed=np.array([18,22,27,33]); expected=np.repeat(observed.sum()/len(observed),len(observed))
plt.figure(figsize=(6,3)); idx=np.arange(4); plt.bar(idx-.18,observed,.36,label="Observed"); plt.bar(idx+.18,expected,.36,label="Expected")
plt.xticks(idx,["A","B","C","D"]); plt.ylabel("Count"); plt.title("Chi-square goodness-of-fit counts"); plt.legend(); plt.show()

## Machine Learning & Forecasting

Introduce foundational supervised learning, clustering, dimensionality reduction, and forecasting methods.

### Decision Trees


In [ ]:
def gini_impurity(labels):
    n=len(labels)
    if n==0: return 0.0
    counts={label:labels.count(label) for label in set(labels)}
    return 1-sum((count/n)**2 for count in counts.values())

print("Gini impurity:", gini_impurity([0,0,0,1,1]))

**SDK visualization**

In [ ]:
X=np.array([[1,1],[1,2],[2,1],[2,2],[4,4],[4,5],[5,4],[5,5]])
y=np.array([0,0,0,0,1,1,1,1]); model=DecisionTreeClassifier(max_depth=2,random_state=7).fit(X,y)
plt.figure(figsize=(8,4)); plot_tree(model,feature_names=["feature_1","feature_2"],class_names=["class 0","class 1"],filled=True); plt.title("Small decision tree classifier"); plt.show()

### K-Means Clustering


In [ ]:
def assign_clusters(points, centroids):
    return [min(range(len(centroids)),key=lambda j:sum((a-b)**2 for a,b in zip(point,centroids[j]))) for point in points]

points=[[1,1],[1,2],[2,1],[8,8],[8,9],[9,8]]; centers=[[1,1],[8,8]]
print("Cluster assignments:", assign_clusters(points,centers))

**SDK visualization**

In [ ]:
rng=np.random.default_rng(4); X=np.vstack([rng.normal([1,1],.5,(50,2)),rng.normal([5,5],.6,(50,2)),rng.normal([9,1],.5,(50,2))])
model=KMeans(n_clusters=3,random_state=4,n_init=10).fit(X)
plt.figure(figsize=(5,4)); plt.scatter(X[:,0],X[:,1],c=model.labels_,s=22); plt.scatter(model.cluster_centers_[:,0],model.cluster_centers_[:,1],marker="X",s=180,edgecolor="black")
plt.title("K-means clusters and centroids"); plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.show()

### Principal Component Analysis (PCA)


In [ ]:
def pca_2d(data):
    means=[sum(col)/len(col) for col in zip(*data)]
    centered=[[row[j]-means[j] for j in range(len(means))] for row in data]
    cov=np.cov(np.array(centered).T); eigenvalues,eigenvectors=np.linalg.eigh(cov)
    order=np.argsort(eigenvalues)[::-1]
    components=eigenvectors[:,order[:2]]
    return np.array(centered) @ components, eigenvalues[order]

projected,eigenvalues=pca_2d([[2,1],[3,2],[4,3],[5,4],[6,5]])
print("Projected coordinates:\n", projected)

**SDK visualization**

In [ ]:
rng=np.random.default_rng(12); X=rng.multivariate_normal([0,0],[ [3,2.4],[2.4,3] ],size=250)
Z, eigvals=pca_2d(X.tolist())
plt.figure(figsize=(5,4)); plt.scatter(Z[:,0],Z[:,1],s=18,alpha=.6); plt.axhline(0,color="gray",lw=.7); plt.axvline(0,color="gray",lw=.7)
plt.xlabel("Principal component 1"); plt.ylabel("Principal component 2"); plt.title("PCA projection"); plt.show()

### Time Series Forecasting


In [ ]:
def moving_average_forecast(values, window):
    if window<1 or len(values)<window: raise ValueError("window must be positive and <= data length")
    return sum(values[-window:])/window

series=[10,12,13,15,14,16,18]
print("Next-step 3-period forecast:", moving_average_forecast(series,3))

**SDK visualization**

In [ ]:
t=np.arange(1,25); series=10+0.4*t+2*np.sin(t/2)+np.random.default_rng(5).normal(0,.5,len(t))
window=3; forecasts=[np.mean(series[i-window:i]) for i in range(window,len(series))]
plt.figure(figsize=(7,3)); plt.plot(t,series,marker="o",label="observed"); plt.plot(t[window:],forecasts,ls="--",marker=".",label="3-period moving average")
plt.xlabel("Time"); plt.ylabel("Value"); plt.title("Time-series moving-average forecast"); plt.legend(); plt.show()

# Probability Distributions & Machine Learning Function Families
### Compact Python demonstrations

Each item follows the same pattern: **what it does → relevance to AI/neural networks → pure Python calculation → SDK example → separate visualization**. The notebook demonstrates individual functions, not complete ML algorithms.

## 1. Probability distributions

### Discrete distributions

These distributions describe countable outcomes. Each distribution below has its own short example and visualization.

In [ ]:
%pip -q install numpy scipy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score, log_loss
)
rng = np.random.default_rng(42)

#### 1. Geometric — (p)

Models the number of trials until the first success. Useful for modeling event counts and waiting processes.

In [ ]:
# Pure Python — evaluate one value
import math
p, k = 0.35, 3
probability = (1-p)**(k-1)*p
print(f"P(first success on trial {k}) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.geom(p=0.35)
x = np.arange(1, 13)
plt.bar(x, d.pmf(x)); plt.title('Geometric distribution (p=0.35)')
plt.xlabel('Trials until first success'); plt.ylabel('Probability'); plt.show()

#### 2. Negative Binomial — (r, p)

Models the number of failures before a fixed number of successes. Can describe over-dispersed count data used in predictive modeling.

In [ ]:
# Pure Python — evaluate one value
import math
r, p, failures = 4, 0.45, 3
probability = math.comb(failures+r-1, r-1)*p**r*(1-p)**failures
print(f"P({failures} failures before {r} successes) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.nbinom(n=4, p=0.45)
x = np.arange(0, 20)
plt.bar(x, d.pmf(x)); plt.title('Negative Binomial (r=4, p=0.45)')
plt.xlabel('Failures'); plt.ylabel('Probability'); plt.show()

#### 3. Hypergeometric — (N, K, n)

Models sampling without replacement. Useful when observations are drawn from a finite population.

In [ ]:
# Pure Python — evaluate one value
import math
N, K, n, successes = 40, 12, 8, 3
probability = math.comb(K,successes)*math.comb(N-K,n-successes)/math.comb(N,n)
print(f"P({successes} successes in sample) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.hypergeom(M=40, n=12, N=8)
x = np.arange(0, 9)
plt.bar(x, d.pmf(x)); plt.title('Hypergeometric (N=40, K=12, n=8)')
plt.xlabel('Successes in sample'); plt.ylabel('Probability'); plt.show()

#### 4. Poisson — (λ)

Models event counts over a fixed interval. Often used for count prediction, arrivals, and incident-rate modeling.

In [ ]:
# Pure Python — evaluate one value
import math
lam, k = 4, 3
probability = math.exp(-lam)*lam**k/math.factorial(k)
print(f"P({k} events) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.poisson(mu=4)
x = np.arange(0, 15)
plt.bar(x, d.pmf(x)); plt.title('Poisson (λ=4)')
plt.xlabel('Event count'); plt.ylabel('Probability'); plt.show()

#### 5. Binomial — (n, p)

Models the number of successes across a fixed number of independent trials. Useful for binary-outcome and reliability modeling.

In [ ]:
# Pure Python — evaluate one value
import math
n, p, k = 12, 0.35, 4
probability = math.comb(n,k)*p**k*(1-p)**(n-k)
print(f"P({k} successes) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.binom(n=12, p=0.35)
x = np.arange(0, 13)
plt.bar(x, d.pmf(x)); plt.title('Binomial (n=12, p=0.35)')
plt.xlabel('Successes'); plt.ylabel('Probability'); plt.show()

#### 6. Bernoulli — (p)

Models one binary event: success or failure. It underlies binary labels and Bernoulli likelihoods in classification.

In [ ]:
# Pure Python — evaluate one value
p, outcome = 0.35, 1
probability = p if outcome == 1 else 1-p
print(f"P(outcome={outcome}) = {probability:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.bernoulli(p=0.35)
x = np.array([0, 1])
plt.bar(x, d.pmf(x)); plt.xticks(x, ['Failure (0)', 'Success (1)'])
plt.title('Bernoulli (p=0.35)'); plt.ylabel('Probability'); plt.show()

### Continuous distributions

These distributions describe continuous-valued quantities. The curves below are probability density functions; probability corresponds to area over an interval.

#### 7. Lognormal — (μ, σ²)

Positive values whose logarithms follow a Normal distribution. Common for skewed quantities such as transaction sizes or durations.

In [ ]:
# Pure Python — evaluate one value
import math
mu, sigma, x = 0, 0.5, 1.5
pdf = math.exp(-(math.log(x)-mu)**2/(2*sigma**2))/(x*sigma*math.sqrt(2*math.pi))
print(f"Lognormal density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.lognorm(s=0.5, scale=np.exp(0))
x = np.linspace(0.01, 5, 400)
plt.plot(x, d.pdf(x)); plt.title('Lognormal (μ=0, σ=0.5)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 8. Normal (Gaussian) — (μ, σ²)

A symmetric bell-shaped distribution. It appears in noise models, statistical assumptions, and parameter initialization.

In [ ]:
# Pure Python — evaluate one value
import math
mu, sigma, x = 1, 1.5, 2
pdf = math.exp(-0.5*((x-mu)/sigma)**2)/(sigma*math.sqrt(2*math.pi))
print(f"Normal density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.norm(loc=1, scale=1.5)
x = np.linspace(-5, 7, 400)
plt.plot(x, d.pdf(x)); plt.title('Normal (μ=1, σ=1.5)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 9. Standard Normal — (0, 1)

The Normal distribution with mean 0 and standard deviation 1. Z-scores and normalized values often use this reference distribution.

In [ ]:
# Pure Python — evaluate one value
import math
z = 1.0
pdf = math.exp(-z*z/2)/math.sqrt(2*math.pi)
print(f"Standard Normal density at z={z} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.norm(loc=0, scale=1)
z = np.linspace(-4, 4, 400)
plt.plot(z, d.pdf(z)); plt.title('Standard Normal N(0, 1)')
plt.xlabel('z-score'); plt.ylabel('Density'); plt.show()

#### 10. Beta — (α, β)

A distribution over values from 0 to 1. Useful for modeling probabilities, rates, and uncertainty about Bernoulli success probabilities.

In [ ]:
# Pure Python — evaluate one value
import math
alpha, beta, x = 2, 5, 0.3
normalizer = math.gamma(alpha+beta)/(math.gamma(alpha)*math.gamma(beta))
pdf = normalizer*x**(alpha-1)*(1-x)**(beta-1)
print(f"Beta density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.beta(a=2, b=5)
x = np.linspace(0, 1, 400)
plt.plot(x, d.pdf(x)); plt.title('Beta (α=2, β=5)')
plt.xlabel('Probability'); plt.ylabel('Density'); plt.show()

#### 11. Gamma — (r, λ)

A positive-valued distribution used for waiting times and rates. It is also a flexible prior in probabilistic models.

In [ ]:
# Pure Python — evaluate one value
import math
shape, scale, x = 2, 2, 3
pdf = x**(shape-1)*math.exp(-x/scale)/(math.gamma(shape)*scale**shape)
print(f"Gamma density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.gamma(a=2, scale=2)  # shape r=2; scale=2 (rate=0.5)
x = np.linspace(0, 16, 400)
plt.plot(x, d.pdf(x)); plt.title('Gamma (shape=2, scale=2)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 12. Uniform — (0, 1)

Every value in the interval has equal density. Uniform sampling is used in simulation and parameter initialization.

In [ ]:
# Pure Python — evaluate one value
a, b, x = 0, 1, 0.4
pdf = 1/(b-a) if a <= x <= b else 0
print(f"Uniform density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.uniform(loc=0, scale=1)
x = np.linspace(-0.2, 1.2, 400)
plt.plot(x, d.pdf(x)); plt.title('Uniform (0, 1)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 13. Cauchy (Standard) — (0, 1)

A heavy-tailed distribution with undefined mean and variance. It is useful as a robust statistical example and caution against relying on averages.

In [ ]:
# Pure Python — evaluate one value
import math
location, scale, x = 0, 1, 1
pdf = 1/(math.pi*scale*(1+((x-location)/scale)**2))
print(f"Cauchy density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.cauchy(loc=0, scale=1)
x = np.linspace(-8, 8, 400)
plt.plot(x, d.pdf(x)); plt.title('Standard Cauchy (0, 1)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 14. Chi-Squared — (ν)

The sum of squared standard Normal variables. It is used in statistical tests and variance-related inference.

In [ ]:
# Pure Python — evaluate one value
import math
nu, x = 4, 3
pdf = x**(nu/2-1)*math.exp(-x/2)/(2**(nu/2)*math.gamma(nu/2))
print(f"Chi-squared density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.chi2(df=4)
x = np.linspace(0, 18, 400)
plt.plot(x, d.pdf(x)); plt.title('Chi-Squared (ν=4)')
plt.xlabel('x'); plt.ylabel('Density'); plt.show()

#### 15. Exponential — (λ)

Models waiting time between events in a Poisson process. Useful for event-time and reliability modeling.

In [ ]:
# Pure Python — evaluate one value
import math
rate, x = 0.5, 2
pdf = rate*math.exp(-rate*x) if x >= 0 else 0
print(f"Exponential density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.expon(scale=2)  # rate λ=0.5
x = np.linspace(0, 12, 400)
plt.plot(x, d.pdf(x)); plt.title('Exponential (λ=0.5)')
plt.xlabel('Waiting time'); plt.ylabel('Density'); plt.show()

#### 16. Weibull — (k, λ)

A flexible lifetime and reliability distribution. Its shape parameter allows increasing, decreasing, or constant hazard rates.

In [ ]:
# Pure Python — evaluate one value
import math
shape, scale, x = 1.5, 2, 3
pdf = (shape/scale)*(x/scale)**(shape-1)*math.exp(-(x/scale)**shape)
print(f"Weibull density at {x} = {pdf:.5f}")

**SciPy + Matplotlib — visualize the same distribution.**

In [ ]:
d = stats.weibull_min(c=1.5, scale=2)
x = np.linspace(0, 8, 400)
plt.plot(x, d.pdf(x)); plt.title('Weibull (k=1.5, scale=2)')
plt.xlabel('Lifetime'); plt.ylabel('Density'); plt.show()

## 2. Machine-learning function families

These are reusable building blocks found across many ML models. Each function is shown independently with a short note about its role in AI or neural-network design.

### 2.1 Activation functions — neural-network architecture

Activation functions transform neuron outputs and introduce nonlinearity. They are an **integral part of neural-network architecture**: without nonlinear activations, stacked linear layers collapse to one linear transformation.

#### ReLU

Common hidden-layer activation; outputs zero for negative inputs and passes positive inputs.

In [ ]:
def relu(x):
    return max(0, x)

print('ReLU(-2) =', relu(-2), '| ReLU(3) =', relu(3))

**Matplotlib visualization:**

In [ ]:
x = np.linspace(-4, 4, 200)
plt.plot(x, relu(x)); plt.title('ReLU'); plt.grid(alpha=.2); plt.show()

#### Sigmoid

Maps values to (0, 1). Often used to produce a binary-classification probability.

In [ ]:
import math

def sigmoid(x):
    return 1/(1+math.exp(-x))

print('Sigmoid(1) =', round(sigmoid(1), 5))

**Matplotlib visualization:**

In [ ]:
x = np.linspace(-8, 8, 200)
plt.plot(x, np.vectorize(sigmoid)(x)); plt.title('Sigmoid'); plt.grid(alpha=.2); plt.show()

#### Tanh

Maps values to (-1, 1). Historically common in recurrent networks and useful when centered activations are desired.

In [ ]:
import math

def tanh(x):
    return math.tanh(x)

print('Tanh(1) =', round(tanh(1), 5))

**Matplotlib visualization:**

In [ ]:
x = np.linspace(-5, 5, 200)
plt.plot(x, np.vectorize(tanh)(x)); plt.title('Tanh'); plt.grid(alpha=.2); plt.show()

#### GELU

Smooth activation used in many Transformer architectures, including language models.

In [ ]:
import math

def gelu(x):
    return 0.5*x*(1+math.tanh(math.sqrt(2/math.pi)*(x+0.044715*x**3)))

print('GELU(1) =', round(gelu(1), 5))

**Matplotlib visualization:**

In [ ]:
x = np.linspace(-5, 5, 200)
plt.plot(x, np.vectorize(gelu)(x)); plt.title('GELU (tanh approximation)'); plt.grid(alpha=.2); plt.show()

#### Softmax

Converts a vector of logits into class probabilities that sum to one; commonly used in multiclass output layers.

In [ ]:
import math

def softmax(values):
    shifted = [v-max(values) for v in values]
    exponentials = [math.exp(v) for v in shifted]
    return [v/sum(exponentials) for v in exponentials]

print('Softmax probabilities:', softmax([2.2, 0.4, -0.7]))

**NumPy SDK equivalent:**

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))  # stable exponentials
    return e / e.sum()

logits = np.array([2.2, 0.4, -0.7])
print(softmax(logits), 'sum =', softmax(logits).sum())

### 2.2 Loss functions — quantify prediction error

A loss function measures the difference between predictions and targets. During neural-network training, gradients of the loss guide parameter updates.

#### Mean Squared Error (MSE)

Squares residuals, penalizing large errors more strongly. Common for regression.

In [ ]:
def mse(y, pred):
    return np.mean((np.asarray(y) - np.asarray(pred))**2)

print(mse([1, 2, 3], [1.2, 1.8, 2.5]))

#### Mean Absolute Error (MAE)

Uses absolute residuals and is less sensitive to large outliers than MSE.

In [ ]:
def mae(y, pred):
    return np.mean(np.abs(np.asarray(y) - np.asarray(pred)))

print(mae([1, 2, 3], [1.2, 1.8, 2.5]))

#### Huber Loss

Quadratic for small errors and linear for large errors; useful when regression targets contain outliers.

In [ ]:
def huber(y, pred, delta=1.0):
    e = np.abs(np.asarray(y) - np.asarray(pred))
    return np.mean(np.where(e <= delta, 0.5*e**2, delta*(e - 0.5*delta)))

print(huber([1, 2, 3], [1.2, 1.8, 5.0]))

#### Binary Cross-Entropy (BCE)

Measures probability error for binary targets. Common with a sigmoid output for binary classification.

In [ ]:
def binary_cross_entropy(y, p, eps=1e-9):
    p = np.clip(p, eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

print(binary_cross_entropy(np.array([0, 1, 1]), np.array([.1, .8, .6])))

#### Categorical Cross-Entropy

Measures probability assigned to the correct class. Common with softmax outputs in multiclass neural networks.

In [ ]:
def categorical_cross_entropy(y_onehot, p, eps=1e-9):
    return -np.mean(np.sum(y_onehot*np.log(np.clip(p, eps, 1)), axis=1))

y = np.array([[0,1,0], [1,0,0]])
p = np.array([[.1,.8,.1], [.7,.2,.1]])
print(categorical_cross_entropy(y, p))

#### Negative Log-Likelihood (NLL)

Penalizes low probability assigned to observed outcomes. A general likelihood-based objective used in probabilistic models.

In [ ]:
def nll(target_index, log_probs):
    return -np.mean(log_probs[np.arange(len(target_index)), target_index])

log_p = np.log([[.1,.8,.1], [.7,.2,.1]])
print(nll(np.array([1, 0]), log_p))

### 2.3 Optimizers — update trainable parameters

Optimizers use gradients to change model parameters. The loss measures error; the optimizer decides how parameters move to reduce it.

#### Gradient Descent / SGD

Moves parameters opposite the gradient. The basic update rule used in neural-network training.

In [ ]:
w = 0.0
learning_rate = 0.1
gradient = 2*(w - 3)  # derivative of (w-3)^2
w -= learning_rate * gradient
print('Updated parameter:', w)

#### Momentum

Adds a running direction to reduce oscillation and accelerate progress along consistent gradients.

In [ ]:
w, velocity = 0.0, 0.0
learning_rate, beta = 0.1, 0.9
gradient = 2*(w - 3)
velocity = beta*velocity + gradient
w -= learning_rate*velocity
print('Updated parameter:', w)

#### RMSProp

Adapts each parameter's learning rate using a moving average of squared gradients; useful for differently scaled gradients.

In [ ]:
w, cache = 0.0, 0.0
learning_rate, beta, eps = 0.01, 0.9, 1e-8
g = 2*(w - 3)
cache = beta*cache + (1-beta)*g**2
w -= learning_rate*g/(np.sqrt(cache)+eps)
print('Updated parameter:', w)

#### Adam

Combines momentum-like first moments and adaptive second moments. Widely used for training deep neural networks and Transformers.

In [ ]:
w, m, v, t = 0.0, 0.0, 0.0, 1
lr, b1, b2, eps = 0.01, 0.9, 0.999, 1e-8
g = 2*(w-3)
m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g**2
mhat = m/(1-b1**t); vhat = v/(1-b2**t)
w -= lr*mhat/(np.sqrt(vhat)+eps)
print('Updated parameter:', w)

### 2.4 Regularization — reduce overfitting

Regularization discourages a model from fitting noise in its training data. These methods affect training behavior and generalization.

#### L1 penalty

Adds the sum of absolute weights to the objective; can encourage sparse parameters.

In [ ]:
weights = np.array([0.2, -1.5, 0.0, 2.0])
lambda_l1 = 0.1
l1 = lambda_l1*np.sum(np.abs(weights))
print('L1 penalty:', l1)

#### L2 penalty / weight decay

Penalizes large squared weights, often encouraging smoother parameter values.

In [ ]:
weights = np.array([0.2, -1.5, 0.0, 2.0])
lambda_l2 = 0.1
l2 = lambda_l2*np.sum(weights**2)
print('L2 penalty:', l2)

#### Dropout

Randomly masks activations during training. It is a neural-network regularization technique, not a separate model.

In [ ]:
activations = np.array([0.5, 1.2, -0.4, 2.0])
keep_prob = 0.8
mask = (rng.random(activations.shape) < keep_prob) / keep_prob
print('Dropout output:', activations*mask)

#### Early stopping

Stops training when validation performance stops improving, helping prevent overfitting.

In [ ]:
validation_loss = [0.9, 0.7, 0.6, 0.62, 0.66]
best_epoch = int(np.argmin(validation_loss))
print('Best epoch:', best_epoch, '| best validation loss:', validation_loss[best_epoch])

### 2.5 Evaluation metrics — measure model performance

Metrics summarize model quality. They are generally used for evaluation rather than directly for gradient-based training.

#### Accuracy

Fraction of predictions that are correct. Can be misleading when classes are imbalanced.

In [ ]:
y = [0, 1, 1, 0]
pred = [0, 1, 0, 0]
print('Accuracy:', accuracy_score(y, pred))

#### Precision

Of the predicted positives, the fraction that are truly positive. Important when false positives are costly.

In [ ]:
y = [0, 1, 1, 0]
pred = [0, 1, 0, 1]
print('Precision:', precision_score(y, pred))

#### Recall

Of all actual positives, the fraction found by the model. Important when false negatives are costly.

In [ ]:
y = [0, 1, 1, 0]
pred = [0, 1, 0, 1]
print('Recall:', recall_score(y, pred))

#### F1 score

Harmonic mean of precision and recall; useful when balancing both matters.

In [ ]:
y = [0, 1, 1, 0]
pred = [0, 1, 0, 1]
print('F1:', f1_score(y, pred))

#### RMSE

Square root of mean squared error; expresses regression error in the target's units.

In [ ]:
y = [2.0, 3.0, 5.0]
pred = [2.5, 2.7, 4.8]
rmse = np.sqrt(mean_squared_error(y, pred))
print('RMSE:', rmse)

#### R-squared (R²)

Measures the proportion of target variance explained by a regression model; can be negative on poor predictions.

In [ ]:
y = [2.0, 3.0, 5.0, 7.0]
pred = [2.5, 2.7, 4.8, 8.0]
print('R²:', r2_score(y, pred))

#### Log Loss

Evaluates predicted probabilities, strongly penalizing confident incorrect predictions.

In [ ]:
y = [0, 1, 1, 0]
p = [0.1, 0.9, 0.4, 0.2]
print('Log loss:', log_loss(y, p))

## 3. Quick reference

| Family | Examples | Role |
|---|---|---|
| Distributions | 16 distributions above | Describe random variables and uncertainty |
| Activations | ReLU, sigmoid, tanh, GELU, softmax | Neural-network architecture and outputs |
| Losses | MSE, MAE, Huber, BCE, categorical CE, NLL | Quantify prediction error |
| Optimizers | SGD, Momentum, RMSProp, Adam | Update trainable parameters |
| Regularization | L1, L2, dropout, early stopping | Reduce overfitting |
| Metrics | Accuracy, precision, recall, F1, RMSE, R², log loss | Evaluate model performance |

**Design note:** every function is intentionally kept small and shown directly beneath its explanation. Larger production implementations may include batching, validation, numerical safeguards, and framework-specific automatic differentiation.